In [1]:
### 프로젝트 루트 경로 설정 및 YAML 설정 파일 로드
# 주피터 노트북이 `notebook/` 폴더 안에서 실행될 때 발생하는 경로 깨짐 문제를 방지하고, 프로젝트 전역 설정(`config/default.yaml`)을 불러옵니다.
import os
import sys
import yaml

# 주피터 노트북 실행 위치에 따른 프로젝트 루트 경로 자동 보정
current_dir = os.getcwd()
if current_dir.endswith("notebook"):
    os.chdir("..")

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# 시스템 전역 설정 파일 로드
with open("config/default.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("설정 로드 완료:", config["generation"])

설정 로드 완료: {'provider': 'openai', 'model': 'gpt-5-nano', 'temperature': 0.1, 'top_p': 0.95, 'max_tokens': 3000}


In [2]:
### 연동 테스트용 Mock 청크 데이터 로드
# 사전에 정의한 가상 제안요청서 청크 데이터(`sample_chunks.jsonl`)를 `SearchResult` 객체로 파싱합니다.
import json
from src.retriever import SearchResult

mock_results = []
file_path = "samples/processed/sample_chunks.jsonl"

# 가상 청크 JSONL 파일을 읽어 SearchResult 객체 리스트로 변환
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        result = SearchResult(
            chunk_id=data["chunk_id"],
            doc_id=data["doc_id"],
            text=data["text"],
            score=0.85,  # 가상 테스트용 임시 유사도 점수
            metadata=data["metadata"]
        )
        mock_results.append(result)

print(f"총 {len(mock_results)}개의 청크 데이터를 성공적으로 불러왔습니다!")

총 3개의 청크 데이터를 성공적으로 불러왔습니다!


In [3]:
### Test Case 1: 문서 내 다중 조건 복합 질문 테스트
# 여러 청크에 나뉘어 있는 정보(주민등록번호 제외 조건, 1년 보관 기간)를 하나의 답변으로 정확히 종합하는지 검증합니다.
from src.rag_engine import generate_answer
question_1 = "기존 시스템에서 사용자 정보 이전할 때 제외해야 하는 항목이 뭐야? 그리고 개인정보 다운로드 기록은 얼마나 보관해야 해?"
response_1 = generate_answer(question_1, mock_results, config)

print(f"Q: {question_1}\n")
print(f"A:\n{response_1['answer']}")

Q: 기존 시스템에서 사용자 정보 이전할 때 제외해야 하는 항목이 뭐야? 그리고 개인정보 다운로드 기록은 얼마나 보관해야 해?

A:
- 기존 시스템에서 사용자 정보 이전 시 제외해야 하는 항목: 주민등록번호 [1].
- 개인정보 조회와 다운로드 기록 보관 기간: 1년 [1].

참고한 문서: sample_rfp.txt [1]


In [4]:
### Test Case 2: 문서 외 질문에 대한 할루시네이션 방어 테스트
# 제공된 문서에 존재하지 않는 기능에 대해 아는 척 지어내지 않고, 단호하게 "확인할 수 없다"고 차단하는지 검증합니다.
question_2 = "이 플랫폼에 블록체인 연계 기능이 포함되어 있어?"
response_2 = generate_answer(question_2, mock_results, config)

print(f"Q: {question_2}\n")
print(f"A:\n{response_2['answer']}")

Q: 이 플랫폼에 블록체인 연계 기능이 포함되어 있어?

A:
- 답변 요지
  - 제공된 문서에서 블록체인 연계 기능에 대한 언급이 없으므로 포함 여부를 확인할 수 없습니다. 이 사실은 문서의 관련 내용에서 확인되며 근거가 되는 부분은 [1][2][3]입니다.

- 추가 안내
  - 블록체인 연계 여부를 확정하려면 발주기관에 직접 확인하시거나, RFP의 보안/연계 항목에 블록체인 관련 요구가 추가로 포함되었는지 재검토하실 것을 권장드립니다.

참고 문서
- [1] sample_rfp.txt
- [2] sample_rfp.txt
- [3] sample_rfp.txt


In [5]:
### Test Case 3: 제안 공고 조건 및 제출 방식 확인 테스트
# RFP의 마감일시와 제한 조건(이메일 제출 불가 등)을 정확하게 추출하여 안내하는지 검증합니다.
question_3 = "제안서 제출 마감일시는 언제이며, 이메일 제출이 가능한가요?"
response_3 = generate_answer(question_3, mock_results, config)

print(f"Q: {question_3}\n")
print(f"A:\n{response_3['answer']}")

Q: 제안서 제출 마감일시는 언제이며, 이메일 제출이 가능한가요?

A:
- 제안서 제출 마감일시: 2026년 8월 14일 17시 [2].
- 이메일 제출 여부: 이메일 제출은 인정되지 않습니다 [2].
- 제출 방법: 나라장터를 통해 온라인으로 제출해야 하며, 전자파일은 PDF 형식으로 등록합니다 [2].

참고한 문서: sample_rfp.txt (2026년 공공 AI 학습지원 플랫폼 구축 사업) [2].


In [6]:
### Test Case 4: 시스템 요구사항 및 제약 조건 추출 테스트
# 문서 내에 명시된 기능적 필수 조건(출처 표시, 근거 없을 시 응답 제한)을 누락 없이 요약하는지 검증합니다.
question_4 = "AI 학습 도우미 기능에서 답변을 제공할 때 지켜야 할 필수 조건은 뭐야?"
response_4 = generate_answer(question_4, mock_results, config)

print(f"Q: {question_4}\n")
print(f"A:\n{response_4['answer']}")

Q: AI 학습 도우미 기능에서 답변을 제공할 때 지켜야 할 필수 조건은 뭐야?

A:
- 필수 조건 1: 질의응답을 지원하는 AI 학습 도우미를 제공하되, 답변에 사용한 교육자료의 제목과 위치를 함께 표시해야 합니다. [1]

- 필수 조건 2: AI 학습 도우미는 검색된 교육자료에 없는 내용을 사실처럼 생성해서는 안 됩니다. [2]

- 필수 조건 3: 근거를 찾지 못하면 확인할 수 없다고 답변해야 합니다. [2]

참고한 문서: sample_rfp.txt


In [7]:
### Test Case 5: 문서에 없는 부가 정책 질문에 대한 방어 테스트
# 예산이나 보안 외의 사내 복리후생 성격의 질문에 대해 추측하지 않고 근거 부재를 올바르게 안내하는지 검증합니다.
question_5 = "이 프로젝트에 투입되는 개발자들의 식대 지원 기준은 어떻게 돼?"
response_5 = generate_answer(question_5, mock_results, config)

print(f"Q: {question_5}\n")
print(f"A:\n{response_5['answer']}")

Q: 이 프로젝트에 투입되는 개발자들의 식대 지원 기준은 어떻게 돼?

A:
제공된 문서에서 근거를 찾을 수 없어 확인할 수 없습니다.

- 다만, 본 RFP에는 개발자 식대 지원 기준이나 식대 정책에 관한 조항이 포함되어 있지 않습니다. 이와 관련한 내용은 제시된 자료 범위 내에서 확인되지 않습니다. [1][2][3]

참고한 문서
- sample_rfp.txt (2026년 공공 AI 학습지원 플랫폼 구축 사업) [1][2][3]


In [8]:
from src.rag_engine import build_context, SYSTEM_RULE
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-5-nano", max_tokens=1000)
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_RULE + "\n\nContext:\n{context}"),
    ("human", "{question}")
])
chain = prompt | llm

msg = chain.invoke({
    "context": build_context(mock_results),
    "question": "이 플랫폼에 블록체인 연계 기능이 포함되어 있어?"
})
print(msg.response_metadata)
print(repr(msg.content))

{'token_usage': {'completion_tokens': 1000, 'prompt_tokens': 1569, 'total_tokens': 2569, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1000, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 1408}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E3I9lMdn41h8O5wT3ebmLSZpqf5O6', 'service_tier': 'default', 'finish_reason': 'length', 'logprobs': None}
''


In [9]:
### End-to-End 연결 검증: profile별 동작 확인
from api_main import run

for profile in ["baseline", "openai", "local"]:
    print(f"=== profile: {profile} ===")
    response = run("사업 예산과 수행 기간은 어떻게 돼?", profile=profile)
    print(response["answer"])
    print(f"(sources: {len(response['sources'])}건)\n")

=== profile: baseline ===
- 제공된 문서들에서 사업 예산에 대한 구체적 금액이나 예산 구성에 관한 정보는 확인되지 않습니다. 예산에 관한 수치나 세부항목은 문서에 기재되어 있지 않습니다 [1][2][3].
- 수행 기간의 구체적 기간도 명시되어 있지 않습니다. 다만 [2]의 경우 계약일로부터 사업완료일까지의 관리 체계와 매주/매월 진척보고 의무를 제시하고 있습니다 [2].
- 참고로 [1]은 보안 교육 등 일정 준수에 관한 내용이지만 전체 사업 기간에 대한 구체적 수치를 제시하지 않습니다 [1]. 또한 [3]은 보안 규정·대책에 대한 내용이 주를 이루며 예산/기간 정보는 포함되어 있지 않습니다 [3].

참고한 문서 출처
- [1] 한국연구재단_2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구구.hwp
- [2] 인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp
- [3] 국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp
(sources: 3건)

=== profile: openai ===
관련 문서 내용을 찾지 못했습니다. 원본 문서나 검색 조건을 다시 확인해 주세요.
(sources: 0건)

=== profile: local ===
'local' 프로필은 아직 준비되지 않았습니다.
사유: Retrieval 2 담당자가 HuggingFace 임베딩 생성, Chroma 저장·조회, SearchResult 변환을 구현해야 합니다.
(sources: 0건)



In [12]:
# --- 셀 1: 정상 검색 → 답변+출처 확인 ---
from api_main import run, print_result
 
question = "한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?"
response = run(question, profile="baseline")
print_result(question, "baseline", response)

[질문] 한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?
[Profile] baseline

[답변]
- 예산: 130,000,000원(VAT 포함) [2]
- 수행 기간: 계약일로부터 3개월이며, 안정화기간 1개월 포함 [2]
- 참고: 기간 및 일정은 학교 사정과 용역대상자와의 협의에 따라 조정될 수 있음 [2]

참고문서: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 제안요청서 (파일명: 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp) [2]

[출처 3건]
1. 한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp | 기관: 한국산업단지공단 | chunk_id: doc_095_chunk_0003 | score: 0.1592
2. 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp | 기관: 한영대학 | chunk_id: doc_001_chunk_0001 | score: 0.1585
3. 한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp | 기관: 한국산업단지공단 | chunk_id: doc_095_chunk_0007 | score: 0.1558


In [11]:
# --- 셀 2: 0건 검색 케이스 확인 ---
question_no_match = "존재하지 않는 사업에 대해 알려줘"
response_no_match = run(
    question_no_match,
    profile="baseline",
    filters={"agency": "존재하지-않는-기관"},
)
print_result(question_no_match, "baseline", response_no_match)

[질문] 존재하지 않는 사업에 대해 알려줘
[Profile] baseline

[답변]
관련 문서 내용을 찾지 못했습니다. 원본 문서나 검색 조건을 다시 확인해 주세요.

[출처 0건]
